# Chatbot with HITL (Human in the Loop)

Agent 처리도중 사람의 개입이 필요한 경우 iterrupt처리할 수 있다.
- 사람의 개입(tool)
- 사람 메세지(command)로 대화를 재개할 수 있다.

In [1]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')

In [2]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(max_results=3)


사람 개입 도구

In [3]:
from langchain_core.tools import tool # Tool 데코레이터
from langgraph.types import interrupt # 사람 응답을 기다리는 interrupt

@tool
def human_assist(query):
    ''' 사람의 개입이 필요하면 이 도구를 사용합니다.'''
    human_response = interrupt({'query': query}) # 사람의 응답이 올때까지 입력 대기
    return human_response['data'] # 사람의 응답 데이터
    
    

In [4]:
from typing import TypedDict, Annotated, List     # 타입 힌트 도구
from langgraph.graph.message import add_messages  # 메시지 누적 처리를 해줄 헬퍼
from langchain.chat_models import init_chat_model 
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage # 메시지 타입
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(max_results = 3)

# 그래프에서 사용할 상태(TypedDict) 스키마 
class State(TypedDict):
    messages: Annotated[List, add_messages] # messages 필드가 add_messages 규칙으로 누적 저장

llm = init_chat_model('gpt-5.6-luna')
tools = [tavily_tool]
llm_with_tools = llm.bind_tools(tools) # llm에 도구 스펙 파인딩



# State의 messages를 입력으로 받아, 응답 메시지를 추가해주는 노드 함수
def chatbot(state: State):   # ← 타입 힌트는 대문자 State
    response = llm.invoke(state['messages']) # 누적된 메시지 리스트로 LLM 호출
    return {'messages': [response]}          # add_messages 규칙에 의해 기존 messages에 응답이 누적됨



In [5]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver() # 상태 체크포인트를 RAM에 저장하는 객체

In [6]:
from pprint import pprint

# 스냅샷 출력 함수 : 저장된 상태(messages)와 다음 실행 노드(next) 확인
def print_snapshot(graph, config):
    snapshot = graph.get_state(config)    # 해당 config(thread_id)의 스냅샷 조회

    if 'messages' in snapshot.values:     # messages가 존재하면
        print(snapshot.values['messages']) # 누적된 messages 출력
    else:
        print('메시지가 없음!')            

    print(f"Next: {snapshot.next}")  # 다음 실행 노드 출력 (튜플)


user1_config = {'configurable': {'thread_id': 'user1'}}

state = {'messages': [('human', '안녕~ 나는 cap이라고 해')]}


In [7]:
from langgraph.graph import StateGraph, START
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver


# 상태 스키마 기반 그래프 생성
builder = StateGraph(MessagesState)


# 도구 실행 노드
tool_node = ToolNode(tools)


# 노드 등록
builder.add_node(
    'chatbot',
    chatbot
)

builder.add_node(
    'tools',
    tool_node
)


# 그래프 연결
builder.add_edge(
    START,
    'chatbot'
)

builder.add_conditional_edges(
    'chatbot',
    tools_condition
)

builder.add_edge(
    'tools',
    'chatbot'
)


# 메모리와 그래프 생성
memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

print('builder 및 graph 생성 완료!')

builder 및 graph 생성 완료!


In [8]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

print('graph 생성 완료!')

graph 생성 완료!


In [9]:
from pprint import pprint
from langgraph.checkpoint.memory import MemorySaver

user1_config = {
    'configurable': {
        'thread_id': 'user1'
    }
}

state = {
    'messages': [
        (
            'human',
            'langgraph를 멀티에이전트 설계에서 사용하는 이유는?'
        )
    ]
}


response = graph.invoke(
    state,
    user1_config
)

pprint(response)

print(
    response['messages'][-1].content
)

{'messages': [HumanMessage(content='langgraph를 멀티에이전트 설계에서 사용하는 이유는?', additional_kwargs={}, response_metadata={}, id='20b9d643-eba7-45d6-880b-3fcae17ed446'),
              AIMessage(content='LangGraph를 멀티에이전트 설계에서 사용하는 가장 큰 이유는 **여러 에이전트의 실행 흐름과 상태를 그래프 형태로 명시적으로 제어할 수 있기 때문**입니다.\n\n주요 이유는 다음과 같습니다.\n\n1. **에이전트 간 역할과 흐름을 명확히 정의**\n   - 리서처 → 분석가 → 작성자처럼 에이전트별 역할을 노드로 표현할 수 있습니다.\n   - 조건에 따라 특정 에이전트로 라우팅하거나 작업을 반복할 수 있습니다.\n   - Supervisor가 작업을 분배하고 결과를 취합하는 구조도 쉽게 구현할 수 있습니다.\n\n2. **공유 상태 관리**\n   - 여러 에이전트가 대화 내용, 작업 결과, 중간 산출물, 오류 정보 등을 공통 상태로 공유할 수 있습니다.\n   - 각 에이전트가 독립적으로 동작하면서도 전체 작업 맥락을 유지할 수 있습니다.\n\n3. **복잡한 워크플로 제어**\n   - 순차 실행뿐 아니라 병렬 실행, 조건 분기, 반복, 재시도 등을 지원합니다.\n   - 단순한 `Agent A → Agent B` 구조를 넘어 다음과 같은 흐름을 만들 수 있습니다.\n\n   ```text\n   요청\n     ↓\n   작업 분류\n     ├─ 검색 에이전트\n     ├─ 계산 에이전트\n     └─ 코딩 에이전트\n     ↓\n   결과 검증\n     ├─ 실패 → 재작업\n     └─ 성공 → 최종 응답\n   ```\n\n4. **장기 실행 및 중단 후 재개**\n   - 실행 상태를 저장해 두었다가 중단된 작업을 이어서 수행할 수 있습니다.\n   - 장시간 걸리는 작업이나 여러 단계의 

In [10]:
from langchain_core.prompts import ChatPromptTemplate


system_prompt = """
**Instruction**
사용자는 케이크 주문, 메뉴 문의, 매장 정보 등을 챗봇에 요청할 수 있다.
챗봇은 친절하고 정확하게 답변한다.

**Context**
당신은 프리미엄 케이크샵의 공식 챗봇이다.
- 운영 시간: 매일 09:00–21:00
- 주요 메뉴: 생크림 케이크, 초코 케이크, 치즈 케이크, 시즌 한정 케이크
- 매장 위치: 서울 강남구 신사동 123-45
- 연락처: 02-1234-5678

**Example**
> **User:** “다음 주 수요일 오후 3시에 케이크 예약 가능한가요?”
> **Bot:** “안녕하세요! 다음 주 수요일(2025년 7월 16일) 오후 3시에 예약 가능합니다. 예약을 원하시면 성함과 연락처를 알려주세요.”

**Output Indicator**
- 챗봇 응답은 한 문단 이내로 간결하게 제시한다.
- 필요한 경우 날짜·시간은 YYYY년 M월 D일 형식으로 명시한다.
- 추가 질문이 필요할 때에는 “추가로 궁금하신 점이 있으면 말씀해주세요.” 로 마무리한다.
- 그외 답변하기 어려운 경우, human_assist 도구를 사용하여 인간의 도움을 요청한다.
"""  # 시스템 지침/컨텍스트/출력 규칙 정의

user_prompt = '당근케익 4호가 홈페이지에서 구매가 안되는데, 매니져님께 구매가능여부를 묻고 싶어요.'  # 사용자 요청 문장

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', '{user_prompt}')
])

state = {'messages': prompt.format_messages(user_prompt = user_prompt)}
response = graph.invoke(state, user1_config)
print(response['messages'][-1].content)

매니저님께 당근케이크 4호 구매 가능 여부를 확인 요청드리겠습니다. 희망하시는 수령 날짜와 시간을 함께 알려주시면 정확히 문의하겠습니다. 추가로 궁금하신 점이 있으면 말씀해주세요.


In [11]:
print_snapshot(graph, user1_config)

[HumanMessage(content='langgraph를 멀티에이전트 설계에서 사용하는 이유는?', additional_kwargs={}, response_metadata={}, id='20b9d643-eba7-45d6-880b-3fcae17ed446'), AIMessage(content='LangGraph를 멀티에이전트 설계에서 사용하는 가장 큰 이유는 **여러 에이전트의 실행 흐름과 상태를 그래프 형태로 명시적으로 제어할 수 있기 때문**입니다.\n\n주요 이유는 다음과 같습니다.\n\n1. **에이전트 간 역할과 흐름을 명확히 정의**\n   - 리서처 → 분석가 → 작성자처럼 에이전트별 역할을 노드로 표현할 수 있습니다.\n   - 조건에 따라 특정 에이전트로 라우팅하거나 작업을 반복할 수 있습니다.\n   - Supervisor가 작업을 분배하고 결과를 취합하는 구조도 쉽게 구현할 수 있습니다.\n\n2. **공유 상태 관리**\n   - 여러 에이전트가 대화 내용, 작업 결과, 중간 산출물, 오류 정보 등을 공통 상태로 공유할 수 있습니다.\n   - 각 에이전트가 독립적으로 동작하면서도 전체 작업 맥락을 유지할 수 있습니다.\n\n3. **복잡한 워크플로 제어**\n   - 순차 실행뿐 아니라 병렬 실행, 조건 분기, 반복, 재시도 등을 지원합니다.\n   - 단순한 `Agent A → Agent B` 구조를 넘어 다음과 같은 흐름을 만들 수 있습니다.\n\n   ```text\n   요청\n     ↓\n   작업 분류\n     ├─ 검색 에이전트\n     ├─ 계산 에이전트\n     └─ 코딩 에이전트\n     ↓\n   결과 검증\n     ├─ 실패 → 재작업\n     └─ 성공 → 최종 응답\n   ```\n\n4. **장기 실행 및 중단 후 재개**\n   - 실행 상태를 저장해 두었다가 중단된 작업을 이어서 수행할 수 있습니다.\n   - 장시간 걸리는 작업이나 여러 단계의 업무 자동화에 적합합니다.\n\n5. **Huma

In [12]:
# 사람 개입
from langgraph.types import Command

human_response = "물론입니다. 구매 가능하시고요. 홈페이지 구매 가능하도록 채널 열어놓겠습니다. 가격은 15만원 입니다."

command = Command(resume={'data': human_response})
response = graph.invoke(command, user1_config)
print(response['messages'][-1].content)

매니저님께 당근케이크 4호 구매 가능 여부를 확인 요청드리겠습니다. 희망하시는 수령 날짜와 시간을 함께 알려주시면 정확히 문의하겠습니다. 추가로 궁금하신 점이 있으면 말씀해주세요.
